# Implement a k-Nearest Neighbors (k-NN) classifier by manually computing pairwise distances, implementing an efficient search algorithm for large datasets, and handling ties in classification decisions.

In [2]:
import pandas as pd
import numpy as np


# 1. Load dataset
df = pd.read_csv("heart.csv")

df = df.dropna(subset=["HeartDisease"]).reset_index(drop=True)

X = df.drop(columns=["HeartDisease"])
y, class_names = pd.factorize(df["HeartDisease"], sort=False)

print("Dataset Shape:", df.shape)
print("Classes:", list(class_names))


# 2. Manually split data into train and test sets
rng = np.random.default_rng(42)

train_indices = []
test_indices = []

for label in np.unique(y):
    indices = np.where(y == label)[0]
    rng.shuffle(indices)

    if len(indices) > 1:
        n_test = max(1, int(len(indices) * 0.20))
        n_test = min(n_test, len(indices) - 1)
    else:
        n_test = 0

    test_indices.extend(indices[:n_test])
    train_indices.extend(indices[n_test:])

train_indices = np.array(train_indices, dtype=int)
test_indices = np.array(test_indices, dtype=int)

rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train = X.iloc[train_indices].copy()
X_test = X.iloc[test_indices].copy()

y_train = y[train_indices]
y_test = y[test_indices]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


# 3. Handle missing values using training data
numeric_cols = [
    col for col in X_train.columns
    if pd.api.types.is_numeric_dtype(X_train[col])
]

categorical_cols = [
    col for col in X_train.columns
    if col not in numeric_cols
]

for col in numeric_cols:
    X_train[col] = pd.to_numeric(
        X_train[col], errors="coerce"
    )
    X_test[col] = pd.to_numeric(
        X_test[col], errors="coerce"
    )

    median_value = X_train[col].median()

    if pd.isna(median_value):
        median_value = 0

    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)


for col in categorical_cols:
    mode_values = X_train[col].mode(dropna=True)

    fill_value = (
        mode_values.iloc[0]
        if not mode_values.empty
        else "Unknown"
    )

    X_train[col] = X_train[col].fillna(fill_value).astype(str)
    X_test[col] = X_test[col].fillna(fill_value).astype(str)


# 4. Encode categorical columns manually through pandas
X_train = pd.get_dummies(
    X_train, columns=categorical_cols, dtype=float
)

X_test = pd.get_dummies(
    X_test, columns=categorical_cols, dtype=float
)

# Keep identical feature columns
X_test = X_test.reindex(
    columns=X_train.columns, fill_value=0
)


# 5. Standardize features
X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)

mean_values = X_train.mean(axis=0)
std_values = X_train.std(axis=0)

std_values[std_values == 0] = 1

X_train = (X_train - mean_values) / std_values
X_test = (X_test - mean_values) / std_values


# 6. Implement k-NN from scratch
class ScratchKNN:

    def __init__(self, k=5, batch_size=128):
        self.k = k
        self.batch_size = batch_size

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)

        if len(self.X_train) == 0:
            raise ValueError("Training data cannot be empty.")

        if self.k < 1:
            raise ValueError("k must be at least 1.")

        self.k = min(self.k, len(self.X_train))

        # Precompute squared training-data norms
        self.train_squared_norms = np.sum(
            self.X_train ** 2, axis=1
        )

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)

        predictions = []

        # Process test data in batches to save memory
        for start in range(0, len(X), self.batch_size):

            batch = X[start:start + self.batch_size]

            # Pairwise squared Euclidean distances:
            # ||a-b||^2 = ||a||^2 + ||b||^2 - 2(a.b)
            batch_squared_norms = np.sum(
                batch ** 2, axis=1
            )

            distances_squared = (
                batch_squared_norms[:, None]
                + self.train_squared_norms[None, :]
                - 2 * (batch @ self.X_train.T)
            )

            # Avoid tiny negative values due to floating-point errors
            np.maximum(distances_squared, 0, out=distances_squared)

            for row_distances in distances_squared:

                # Efficiently find the k nearest candidates
                candidate_indices = np.argpartition(
                    row_distances, self.k - 1
                )[:self.k]

                # Find the distance of the kth neighbor
                kth_distance = np.max(
                    row_distances[candidate_indices]
                )

                # Include all candidates tied at the kth distance
                candidates = np.flatnonzero(
                    row_distances <= kth_distance + 1e-12
                )

                # Sort by distance, then training row index
                order = np.lexsort((
                    candidates,
                    row_distances[candidates]
                ))

                neighbors = candidates[order[:self.k]]

                neighbor_labels = self.y_train[neighbors]

                neighbor_distances = np.sqrt(
                    row_distances[neighbors]
                )

                # Count votes for each class
                classes, counts = np.unique(
                    neighbor_labels, return_counts=True
                )

                max_votes = counts.max()
                tied_classes = classes[counts == max_votes]

                # If one class has the most votes, select it
                if len(tied_classes) == 1:
                    prediction = tied_classes[0]

                else:
                    # Tie-break 1: smaller total neighbor distance
                    distance_sums = {}

                    for label in tied_classes:
                        distance_sums[label] = neighbor_distances[
                            neighbor_labels == label
                        ].sum()

                    best_distance = min(distance_sums.values())

                    finalists = [
                        label for label in tied_classes
                        if np.isclose(
                            distance_sums[label],
                            best_distance,
                            rtol=1e-12,
                            atol=1e-12
                        )
                    ]

                    # Tie-break 2: closest neighbor among finalists
                    prediction = next(
                        label for label in neighbor_labels
                        if label in finalists
                    )

                predictions.append(prediction)

        return np.asarray(predictions)


# 7. Train and predict
knn = ScratchKNN(k=5, batch_size=128)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

print("\nk-NN prediction completed.")


# 8. Calculate evaluation metrics manually
accuracy = np.mean(y_test == y_pred)

print("\n--- k-NN Evaluation ---")
print(f"Accuracy: {accuracy:.4f}")


# Class-wise precision, recall and F1-score
print("\nClass-wise Results:")

for class_id, class_name in enumerate(class_names):

    TP = np.sum((y_test == class_id) & (y_pred == class_id))
    FP = np.sum((y_test != class_id) & (y_pred == class_id))
    FN = np.sum((y_test == class_id) & (y_pred != class_id))

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    print(f"\nClass: {class_name}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")


# 9. Confusion matrix manually
n_classes = len(class_names)

confusion_matrix = np.zeros(
    (n_classes, n_classes), dtype=int
)

for actual, predicted in zip(y_test, y_pred):
    confusion_matrix[actual, predicted] += 1

print("\nConfusion Matrix:")
print(confusion_matrix)


# 10. Try different k values
print("\n--- Comparison of Different k Values ---")

for k_value in [1, 3, 5, 7, 9]:

    model = ScratchKNN(
        k=k_value,
        batch_size=128
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    acc = np.mean(predictions == y_test)

    print(f"k = {k_value}: Accuracy = {acc:.4f}")

Dataset Shape: (918, 12)
Classes: [0, 1]
Training rows: 735
Testing rows: 183

k-NN prediction completed.

--- k-NN Evaluation ---
Accuracy: 0.8251

Class-wise Results:

Class: 0
Precision: 0.8378
Recall:    0.7561
F1-score:  0.7949

Class: 1
Precision: 0.8165
Recall:    0.8812
F1-score:  0.8476

Confusion Matrix:
[[62 20]
 [12 89]]

--- Comparison of Different k Values ---
k = 1: Accuracy = 0.7978
k = 3: Accuracy = 0.8033
k = 5: Accuracy = 0.8251
k = 7: Accuracy = 0.8142
k = 9: Accuracy = 0.8033
